In [4]:
# =============================================================================
# Notebook : 02_feature_engineering.ipynb
# Project  : Titanic Survival Prediction
# Phase    : Feature Engineering
#
# Objective
# ---------
# Transform raw passenger information into features that better represent the
# underlying patterns in the data.
#
# Feature engineering is the process of creating new information from existing
# data so that machine learning models can learn more effectively.
#
# Expected Outcome
# ----------------
# - Create FamilySize
# - Create IsAlone
# - Extract Passenger Titles
# - Extract Cabin Deck
# =============================================================================

In [5]:
# -----------------------------------------------------------------------------
# WHY:
#
# Import the libraries required for feature engineering.
#
# We also configure pandas so that all columns are visible during inspection.
# -----------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [6]:
# -----------------------------------------------------------------------------
# WHY:
#
# Every notebook should be reproducible.
#
# Reload the raw datasets instead of relying on variables created in previous
# notebooks.
#
# Raw datasets should never be modified directly.
# -----------------------------------------------------------------------------

train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [7]:
# -----------------------------------------------------------------------------
# WHY:
#
# SibSp and Parch individually describe only part of a passenger's family.
#
# Combining them creates a more informative feature representing the total
# number of family members travelling together.
#
# We add one because the passenger themselves are also part of the family.
#
# This is our first engineered feature.
# -----------------------------------------------------------------------------

train_df["FamilySize"] = (
    train_df["SibSp"] +
    train_df["Parch"] +
    1
)

test_df["FamilySize"] = (
    test_df["SibSp"] +
    test_df["Parch"] +
    1
)

In [8]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# What is the distribution of the newly created FamilySize feature?
#
# WHY:
#
# Before using a new feature, we should understand its distribution and check
# for unusual values.
# -----------------------------------------------------------------------------

train_df["FamilySize"].value_counts().sort_index()

FamilySize
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64

In [9]:
# -----------------------------------------------------------------------------
# WHY:
#
# Many machine learning models perform better when a meaningful binary feature
# replaces a more granular numeric feature.
#
# This feature identifies whether the passenger was travelling alone.
# -----------------------------------------------------------------------------

train_df["IsAlone"] = (
    train_df["FamilySize"] == 1
).astype(int)

test_df["IsAlone"] = (
    test_df["FamilySize"] == 1
).astype(int)

In [10]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# How many passengers were travelling alone?
#
# WHY:
#
# This helps us understand whether travelling alone was common.
# -----------------------------------------------------------------------------

train_df["IsAlone"].value_counts()

IsAlone
1    537
0    354
Name: count, dtype: int64

In [11]:
# -----------------------------------------------------------------------------
# WHY:
#
# The passenger's full name is difficult for a machine learning model to use
# directly.
#
# However, titles such as Mr, Mrs, Miss and Master contain structured
# information that may correlate with survival.
#
# We extract only the title from the Name column.
# -----------------------------------------------------------------------------

train_df["Title"] = train_df["Name"].str.extract(
    r",\s*([^\.]+)\."
)

test_df["Title"] = test_df["Name"].str.extract(
    r",\s*([^\.]+)\."
)

In [12]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# Which passenger titles occur most frequently?
#
# WHY:
#
# Rare titles may later be grouped together to reduce the number of categories.
# -----------------------------------------------------------------------------

train_df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

# =============================================================================
# Engineering Decision Card
# =============================================================================

Feature
-------
Passenger Title

Why create it?
--------------
The passenger's title contains structured information such as gender,
age group and social status which may influence survival.

Observations
------------
- Four titles account for nearly all passengers.
- Most remaining titles occur only one or two times.
- Rare categories provide insufficient data for reliable learning.

Decision
--------
Keep:
- Mr
- Mrs
- Miss
- Master

Group every remaining title into a single category:
- Rare

Reason
------
Reducing cardinality improves generalization and reduces the likelihood of
overfitting to categories with very few samples.

In [14]:
# =============================================================================
# FEATURE: Group Rare Passenger Titles
# =============================================================================

# -----------------------------------------------------------------------------
# WHY:
#
# Extremely infrequent categories increase feature cardinality and can cause
# the model to overfit to categories with only a handful of observations.
#
# We retain the four most common titles and group all remaining titles into
# a single category called "Rare".
# -----------------------------------------------------------------------------

COMMON_TITLES = [
    "Mr",
    "Mrs",
    "Miss",
    "Master"
]

In [15]:
# -----------------------------------------------------------------------------
# WHY:
#
# Replace every uncommon title with the category "Rare".
#
# This reduces the number of categories while preserving the information that
# a passenger had an uncommon social title.
# -----------------------------------------------------------------------------

train_df["Title"] = train_df["Title"].apply(
    lambda title: title if title in COMMON_TITLES else "Rare"
)

test_df["Title"] = test_df["Title"].apply(
    lambda title: title if title in COMMON_TITLES else "Rare"
)

In [16]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# Did the transformation produce the expected categories?
#
# WHY:
#
# Every preprocessing step should be verified immediately after it is applied.
# -----------------------------------------------------------------------------

train_df["Title"].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64

In [17]:
# -----------------------------------------------------------------------------
# WHY:
#
# The Cabin feature contains two pieces of information:
#
#     Cabin = Deck + Cabin Number
#
# Example:
#
#     C85
#
# becomes
#
#     Deck = C
#
# We hypothesize that the deck on which a passenger stayed may influence
# survival probability more than the exact cabin number.
#
# Passengers on different decks may have had:
#   Different proximity to lifeboats
#   Different ticket classes
#   Different accessibility
#   Different evacuation times
# -----------------------------------------------------------------------------
train_df['Deck'] = train_df["Cabin"].str[0]
test_df['Deck'] = test_df["Cabin"].str[0]


In [19]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# What are the unique deck values?
#
# WHY:
#
# Inspect the extracted feature before deciding how to preprocess it.
# -----------------------------------------------------------------------------
train_df['Deck'].value_counts(dropna=False)

Deck
NaN    687
C       59
B       47
D       33
E       32
A       15
F       13
G        4
T        1
Name: count, dtype: int64

# =============================================================================
# Engineering Decision Card
# =============================================================================

Feature
-------
Deck

Why create it?
--------------
The deck may capture a passenger's physical location on the ship.

Observations
------------
- Cabin contains many missing values.
- Deck extraction greatly reduces the number of categories.
- Missing values still remain.

Decision
--------
Keep Deck for now.

Reason
------
Even though many values are missing, the available deck information may contain
valuable predictive signal.

In [20]:
# -----------------------------------------------------------------------------
# WHY:
#
# Before engineering features, inspect the raw values.
#
# We are looking for:
#   - Prefixes
#   - Numeric-only tickets
#   - Formatting inconsistencies
#
# Understanding the raw format helps us engineer meaningful features instead of
# making assumptions.
# -----------------------------------------------------------------------------

train_df["Ticket"].sample(20, random_state=42)

709                2661
439          C.A. 18723
840    SOTON/O2 3101287
720              248727
39                 2651
290               19877
300                9234
333              345764
208              367231
136               11752
137              113803
696              363592
485                4133
244                2694
344              229236
853            PC 17592
621               11753
653              330919
886              211536
110              110465
Name: Ticket, dtype: str

In [22]:
# -----------------------------------------------------------------------------
# WHY:
#
# Ticket prefixes may represent ticket categories, issuing offices or booking
# groups.
#
# Passengers with similar prefixes may share travel characteristics.
#
# Numeric-only tickets receive the category "NONE" rather than NaN because
# the ticket exists—it simply has no alphabetic prefix.
# -----------------------------------------------------------------------------
train_df['TicketPrefix'] = (train_df['Ticket']
    .str.replace(r"[0-9]", "", regex=True)
    .str.strip()
    .str.replace(r"[./]", "", regex=True)
)

train_df["TicketPrefix"] = train_df["TicketPrefix"].replace("", "NONE")

test_df['TicketPrefix'] = (test_df['Ticket']
    .str.replace(r"[0-9]", "", regex=True)
    .str.strip()
    .str.replace(r"[./]", "", regex=True)
)

test_df["TicketPrefix"] = test_df["TicketPrefix"].replace("", "NONE")

In [23]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# Which ticket prefixes are most common?
#
# WHY:
#
# Understanding the distribution helps us decide whether some prefixes should
# later be grouped into a "Rare" category.
# -----------------------------------------------------------------------------

train_df["TicketPrefix"].value_counts()

TicketPrefix
NONE          661
PC             60
CA             41
A              28
SOTONOQ        15
STONO          12
WC             10
SCPARIS         7
STONO           6
SOC             6
C               5
FCC             5
SCParis         4
LINE            4
PP              3
WEP             3
SOPP            3
SWPP            2
PPP             2
SCAH            2
SOTONO          2
SCA             1
SP              1
SOP             1
Fa              1
SCOW            1
SC              1
AS              1
SCAH Basle      1
FC              1
CASOTON         1
Name: count, dtype: int64

In [26]:
# -----------------------------------------------------------------------------
# WHY:
#
# Passengers sharing the same ticket are likely travelling together.
#
# The number of passengers sharing a ticket may provide additional information
# beyond FamilySize.
#
# We count how many times each ticket appears in the dataset.
# -----------------------------------------------------------------------------
ticket_group_size = train_df.groupby("Ticket")["Ticket"].transform("count")
train_df['TicketGroupSize'] = ticket_group_size

ticket_group_size = test_df.groupby("Ticket")["Ticket"].transform("count")
test_df["TicketGroupSize"] = ticket_group_size

In [27]:
# -----------------------------------------------------------------------------
# QUESTION:
#
# How many passengers share the same ticket?
#
# WHY:
#
# Understanding the distribution helps determine whether travelling together
# provides additional predictive signal.
# -----------------------------------------------------------------------------

train_df["TicketGroupSize"].value_counts().sort_index()

TicketGroupSize
1    547
2    188
3     63
4     44
5     10
6     18
7     21
Name: count, dtype: int64